In [186]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import pickle
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
import logging
import pandas as pd

In [187]:
fbref_url = "https://fbref.com{}"
premier_league = fbref_url.format("/en/comps/9/Premier-League-Stats")
data = requests.get(premier_league)

# CHANGE DETAILS BELOW FOR PERSONAL INSTANCE WHEN RUNNING 

In [188]:
# alex instance details 
NEO4J_URI='neo4j+s://14544b1e.databases.Sneo4j.io'
NEO4J_USERNAME='neo4j'
NEO4J_PASSWORD='DSB0RkgvGa1atsHc5KoiVPst4l95qg_iMADHrRDXmzA'


In [220]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

In [190]:
with driver.session(database="neo4j") as session:
    query = """
    MATCH (n)
    DETACH DELETE n
    """
    session.run(query)

In [191]:
with driver.session(database="neo4j") as session:
    query = "CREATE (l:League {name: $name})"
    session.run(query, {"name":"PremierLegue24-25"})

In [192]:
with open(f'team_dict_24-25.pkl', 'rb') as f:
        team_dict = pickle.load(f)

In [193]:
team_players = dict()
player_details = list()
for team, players in team_dict.items():
    team_players[team] = list()
    for player in players.keys():
        team_players[team].append(player)
        players[player]['Name'] = player
        player_details.append(players[player])

In [194]:
with driver.session(database="neo4j") as session:
    query = """
            UNWIND $players AS player
            MERGE (p:Player {name: player.Name})
            SET p.position = player.position,
                p.footed = player.footed,
                p.height = player.height,
                p.weight = player.weight,
                p.dob = player.dob,
                p.birth_place = player.birth_place,
                p.national_team = player.national_team,
                p.club = player.club,
                p.wages = player.wages,
                p.contract_expiry = player.contract_expiry,
                p.img = player.img
    """
    
    session.run(query, players=player_details)

In [195]:
# clean up before sending
with open('match_reports.pkl','rb') as f:
    match_reports = pickle.load(f)

In [196]:
fixtures_df = pd.read_csv('fixtures.csv')
fixtures_df[['Home_Score', 'Away_Score']] = fixtures_df['Score'].str.split('–', expand=True)

In [197]:
all_players_dict = pd.read_csv('all_players.csv', index_col=0).to_dict(orient='index')

In [198]:
# this is very important please do not delete
mappings = {"Brighton":'Brighton & Hove Albion',
            'Manchester Utd':'Manchester United',
            'Newcastle Utd':'Newcastle United',
            "Nott'ham Forest":'Nottingham Forest',
            'Tottenham':'Tottenham Hotspur',
            'West Ham':'West Ham United',
            'Wolves':'Wolverhampton Wanderers'}

In [199]:
# fixtures_df['Home'] = fixtures_df['Home'].map(mappings)
# fixtures_df['Away'] = fixtures_df['Away'].map(mappings)
fixtures_df['Home'] = fixtures_df['Home'].apply(lambda x: mappings[x] if x in mappings else x)
fixtures_df['Away'] = fixtures_df['Away'].apply(lambda x: mappings[x] if x in mappings else x)



In [200]:
with driver.session(database="neo4j") as session:
    query = "match (n) detach delete n"
    session.run(query)

In [201]:
with driver.session(database="neo4j") as session:
    query = "CREATE (l:League {name: $name})"
    session.run(query, {"name":"PremierLeague"})

    query = "CREATE (s:Season {name: $name})"
    session.run(query, {"name":"24-25"})

    query = """
        MATCH (s:Season{name:$season_name}), (l:League{name: $league_name}) 
        CREATE (s)-[:SEASON_OF_LEAGUE]->(l)
        """
    print(session.run(query, {"league_name": "PremierLeague", "season_name": "24-25"}))

In [202]:
team_players = dict()
player_details = list()

for team, players in team_dict.items():
    team_players[team] = list()
    for player in players.keys():
        team_players[team].append(player)
        players[player]['Name'] = player
        players[player]['Team'] = team  # Fix the key access from player to players
        player_details.append(players[player])
    team_players[team] = team_players[team]

with driver.session(database="neo4j") as session:
    query = """
            UNWIND $players AS player
            MERGE (p:Player {name: player.Name})
            SET p.position = player.position,
                p.footed = player.footed,
                p.height = player.height,
                p.weight = player.weight,
                p.dob = player.dob,
                p.birth_place = player.birth_place,
                p.national_team = player.national_team,
                p.club = player.club,
                p.wages = player.wages,
                p.contract_expiry = player.contract_expiry,
                p.img = player.img

            MERGE (team:Team {name: player.Team})
            MERGE (p)-[:PLAYS_FOR]->(team)
    """
    
    session.run(query, players=player_details)


In [203]:
with driver.session(database="neo4j") as session:
    # Create GameWeek nodes and relationships
    for gw in range(1, 39):  # Loop from 1 to 38
        query = """
        MATCH (s:Season {name: $season_name})
        CREATE (gw:GameWeek {number: $number})-[:IS_A_GW]->(s)
        """
        session.run(query, {"season_name": "24-25", "number": gw})

In [204]:
print(fixtures_df.isna().sum())


Unnamed: 0        0
GW                0
Day               0
Date              0
Time              0
Home              0
xG              173
Score           173
xG.1            173
Away              0
Attendance      173
Venue             0
Referee         173
Match Report      0
Notes           380
Home_Score      173
Away_Score      173
dtype: int64


In [205]:
with driver.session(database="neo4j") as session:
    for _, row in fixtures_df.iterrows():
        query = """
        MERGE (match:Match {id: $match_id})
        SET match.date = $date,
            match.time = $time,
            match.home_team = $home_team,
            match.away_team = $away_team,
            match.home_expected = $home_expected,
            match.away_expected = $away_expected,
            match.home_score = $home_score,
            match.away_score = $away_score

        MERGE (ref:Referee {name: $referee})
        MERGE (venue:Venue {name: $venue})
        
        MERGE (home_team:Team {name: $home_team})
        MERGE (away_team:Team {name: $away_team})
        
        MERGE (match)-[:REFERRED_BY]->(ref)
        MERGE (match)-[:PLAYED_AT]->(venue)
        MERGE (match)-[:HOME_TEAM]->(home_team)
        MERGE (match)-[:AWAY_TEAM]->(away_team)
        """
        
        # Prepare the data dictionary
        data ={
            "match_id": f"GW{row['GW']}_{row['Home']}_vs_{row['Away']}",
            "date": row["Date"],
            "time": row["Time"],
            "home_team": row["Home"],
            "away_team": row["Away"],
        }
        if row["Match Report"]:
            data["home_expected"]= row["xG"],
            data["away_expected"]= row["xG.1"],
            data["home_score"]= row["Home_Score"],
            data["away_score"]= row["Away_Score"],
            data["referee"]= row["Referee"],
            data["venue"] =  row["Venue"]
        
        # Execute the query
        session.run(query, data)


In [206]:
with driver.session(database="neo4j") as session:
    for i, row in fixtures_df[:len(match_reports)].iterrows():
        match_id = f"GW{row['GW']}_{row['Home']}_vs_{row['Away']}"


        home_team_stats = match_reports[i][0][0][:-1] 
        away_team_stats = match_reports[i][1][0][:-1] 
        home_team_goaly_stats = match_reports[i][0][1] 
        away_team_goaly_stats = match_reports[i][1][1]

        home_team_name = row['Home']
        away_team_name = row['Away']


        # Add Home Team Player Stats
        for _, player_stats in home_team_stats.iterrows():
            player_query = """
            MERGE (player:Player {name: $player_name, nation: $nation, position: $position, age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (player)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[h:HOME_TEAM_PLAYER_STATS]->(player)
            SET h.goals = $goals,
                h.assists = $assists,
                h.penalties = $penalties,
                h.shots = $shots,
                h.shots_on_target = $shots_on_target,
                h.touches = $touches,
                h.tackles = $tackles,
                h.interceptions = $interceptions,
                h.blocks = $blocks,
                h.xg = $xg,
                h.npxg = $npxg,
                h.xag = $xag,
                h.sca = $sca,
                h.gca = $gca,
                h.passes_completed = $passes_completed,
                h.passes_attempted = $passes_attempted,
                h.pass_accuracy = $pass_accuracy,
                h.progressive_passes = $progressive_passes,
                h.carries = $carries,
                h.progressive_carries = $progressive_carries
            """
            session.run(player_query, {
                "player_name": player_stats[('Unnamed: 0_level_0', 'Player')],
                "nation": player_stats[('Unnamed: 2_level_0', 'Nation')],
                "position": player_stats[('Unnamed: 3_level_0', 'Pos')],
                "age": player_stats[('Unnamed: 4_level_0', 'Age')],
                "match_id": match_id,
                "team_name": home_team_name,
                "goals": player_stats[('Performance', 'Gls')],
                "assists": player_stats[('Performance', 'Ast')],
                "penalties": player_stats[('Performance', 'PK')],
                "shots": player_stats[('Performance', 'Sh')],
                "shots_on_target": player_stats[('Performance', 'SoT')],
                "touches": player_stats[('Performance', 'Touches')],
                "tackles": player_stats[('Performance', 'Tkl')],
                "interceptions": player_stats[('Performance', 'Int')],
                "blocks": player_stats[('Performance', 'Blocks')],
                "xg": player_stats[('Expected', 'xG')],
                "npxg": player_stats[('Expected', 'npxG')],
                "xag": player_stats[('Expected', 'xAG')],
                "sca": player_stats[('SCA', 'SCA')],
                "gca": player_stats[('SCA', 'GCA')],
                "passes_completed": player_stats[('Passes', 'Cmp')],
                "passes_attempted": player_stats[('Passes', 'Att')],
                "pass_accuracy": player_stats[('Passes', 'Cmp%')],
                "progressive_passes": player_stats[('Passes', 'PrgP')],
                "carries": player_stats[('Carries', 'Carries')],
                "progressive_carries": player_stats[('Carries', 'PrgC')],
            })

        # Add Away Team Player Stats
        for _, player_stats in away_team_stats.iterrows():
            player_query = """
            MERGE (player:Player {name: $player_name, nation: $nation, position: $position, age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (player)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[a:AWAY_TEAM_PLAYER_STATS]->(player)
            SET a.goals = $goals,
                a.assists = $assists,
                a.penalties = $penalties,
                a.shots = $shots,
                a.shots_on_target = $shots_on_target,
                a.touches = $touches,
                a.tackles = $tackles,
                a.interceptions = $interceptions,
                a.blocks = $blocks,
                a.xg = $xg,
                a.npxg = $npxg,
                a.xag = $xag,
                a.sca = $sca,
                a.gca = $gca,
                a.passes_completed = $passes_completed,
                a.passes_attempted = $passes_attempted,
                a.pass_accuracy = $pass_accuracy,
                a.progressive_passes = $progressive_passes,
                a.carries = $carries,
                a.progressive_carries = $progressive_carries
            """
            session.run(player_query, {
                "player_name": player_stats[('Unnamed: 0_level_0', 'Player')],
                "nation": player_stats[('Unnamed: 2_level_0', 'Nation')],
                "position": player_stats[('Unnamed: 3_level_0', 'Pos')],
                "age": player_stats[('Unnamed: 4_level_0', 'Age')],
                "match_id": match_id,
                "team_name": away_team_name,
                "goals": player_stats[('Performance', 'Gls')],
                "assists": player_stats[('Performance', 'Ast')],
                "penalties": player_stats[('Performance', 'PK')],
                "shots": player_stats[('Performance', 'Sh')],
                "shots_on_target": player_stats[('Performance', 'SoT')],
                "touches": player_stats[('Performance', 'Touches')],
                "tackles": player_stats[('Performance', 'Tkl')],
                "interceptions": player_stats[('Performance', 'Int')],
                "blocks": player_stats[('Performance', 'Blocks')],
                "xg": player_stats[('Expected', 'xG')],
                "npxg": player_stats[('Expected', 'npxG')],
                "xag": player_stats[('Expected', 'xAG')],
                "sca": player_stats[('SCA', 'SCA')],
                "gca": player_stats[('SCA', 'GCA')],
                "passes_completed": player_stats[('Passes', 'Cmp')],
                "passes_attempted": player_stats[('Passes', 'Att')],
                "pass_accuracy": player_stats[('Passes', 'Cmp%')],
                "progressive_passes": player_stats[('Passes', 'PrgP')],
                "carries": player_stats[('Carries', 'Carries')],
                "progressive_carries": player_stats[('Carries', 'PrgC')],
            })

         # Add Home Team Goalie Stats
        for _, goalie_stats in home_team_goaly_stats.iterrows():
            goalie_query = """
            MERGE (goalie:Player {name: $goalie_name, nation: $nation, position: "GK", age: $age})
            MERGE (team:Team {name: $team_name})
            MERGE (goalie)-[:PLAYS_FOR]->(team)
            MERGE (match:Match {id: $match_id})
            MERGE (match)-[:HOME_TEAM_GOALIE_STATS]->(goalie)
            SET goalie.minutes_played = $minutes_played,
                goalie.sota = $sota,
                goalie.goals_allowed = $goals_allowed,
                goalie.saves = $saves,
                goalie.save_percentage = $save_percentage,
                goalie.psxg = $psxg,
                goalie.passes_completed = $passes_completed,
                goalie.passes_attempted = $passes_attempted,
                goalie.pass_accuracy = $pass_accuracy,
                goalie.gk_passes_attempted = $gk_passes_attempted,
                goalie.throws = $throws,
                goalie.launch_percentage = $launch_percentage,
                goalie.launch_average_length = $launch_average_length,
                goalie.goal_kicks_attempted = $goal_kicks_attempted,
                goalie.goal_kick_launch_percentage = $goal_kick_launch_percentage,
                goalie.goal_kick_average_length = $goal_kick_average_length,
                goalie.crosses_opportunities = $crosses_opportunities,
                goalie.crosses_stops = $crosses_stops,
                goalie.crosses_stop_percentage = $crosses_stop_percentage,
                goalie.opa = $opa,
                goalie.opa_average_distance = $opa_average_distance
            """
            session.run(goalie_query, {
                "goalie_name": goalie_stats[('Unnamed: 0_level_0', 'Player')],
                "nation": goalie_stats[('Unnamed: 1_level_0', 'Nation')],
                "age": goalie_stats[('Unnamed: 2_level_0', 'Age')],
                "match_id": match_id,
                "team_name": home_team_name,
                "minutes_played": goalie_stats[('Unnamed: 3_level_0', 'Min')],
                "sota": goalie_stats[('Shot Stopping', 'SoTA')],
                "goals_allowed": goalie_stats[('Shot Stopping', 'GA')],
                "saves": goalie_stats[('Shot Stopping', 'Saves')],
                "save_percentage": goalie_stats[('Shot Stopping', 'Save%')],
                "psxg": goalie_stats[('Shot Stopping', 'PSxG')],
                "passes_completed": goalie_stats[('Launched', 'Cmp')],
                "passes_attempted": goalie_stats[('Launched', 'Att')],
                "pass_accuracy": goalie_stats[('Launched', 'Cmp%')],
                "gk_passes_attempted": goalie_stats[('Passes', 'Att (GK)')],
                "throws": goalie_stats[('Passes', 'Thr')],
                "launch_percentage": goalie_stats[('Passes', 'Launch%')],
                "launch_average_length": goalie_stats[('Passes', 'AvgLen')],
                "goal_kicks_attempted": goalie_stats[('Goal Kicks', 'Att')],
                "goal_kick_launch_percentage": goalie_stats[('Goal Kicks', 'Launch%')],
                "goal_kick_average_length": goalie_stats[('Goal Kicks', 'AvgLen')],
                "crosses_opportunities": goalie_stats[('Crosses', 'Opp')],
                "crosses_stops": goalie_stats[('Crosses', 'Stp')],
                "crosses_stop_percentage": goalie_stats[('Crosses', 'Stp%')],
                "opa": goalie_stats[('Sweeper', '#OPA')],
                "opa_average_distance": goalie_stats[('Sweeper', 'AvgDist')],
            })

        # Add Away Team Goalie Stats
        for _, goalie_stats in away_team_goaly_stats.iterrows():
            session.run(goalie_query, {
                "goalie_name": goalie_stats[('Unnamed: 0_level_0', 'Player')],
                "nation": goalie_stats[('Unnamed: 1_level_0', 'Nation')],
                "age": goalie_stats[('Unnamed: 2_level_0', 'Age')],
                "match_id": match_id,
                "team_name": away_team_name,
                "minutes_played": goalie_stats[('Unnamed: 3_level_0', 'Min')],
                "sota": goalie_stats[('Shot Stopping', 'SoTA')],
                "goals_allowed": goalie_stats[('Shot Stopping', 'GA')],
                "saves": goalie_stats[('Shot Stopping', 'Saves')],
                "save_percentage": goalie_stats[('Shot Stopping', 'Save%')],
                "psxg": goalie_stats[('Shot Stopping', 'PSxG')],
                "passes_completed": goalie_stats[('Launched', 'Cmp')],
                "passes_attempted": goalie_stats[('Launched', 'Att')],
                "pass_accuracy": goalie_stats[('Launched', 'Cmp%')],
                "gk_passes_attempted": goalie_stats[('Passes', 'Att (GK)')],
                "throws": goalie_stats[('Passes', 'Thr')],
                "launch_percentage": goalie_stats[('Passes', 'Launch%')],
                "launch_average_length": goalie_stats[('Passes', 'AvgLen')],
                "goal_kicks_attempted": goalie_stats[('Goal Kicks', 'Att')],
                "goal_kick_launch_percentage": goalie_stats[('Goal Kicks', 'Launch%')],
                "goal_kick_average_length": goalie_stats[('Goal Kicks', 'AvgLen')],
                "crosses_opportunities": goalie_stats[('Crosses', 'Opp')],
                "crosses_stops": goalie_stats[('Crosses', 'Stp')],
                "crosses_stop_percentage": goalie_stats[('Crosses', 'Stp%')],
                "opa": goalie_stats[('Sweeper', '#OPA')],
                "opa_average_distance": goalie_stats[('Sweeper', 'AvgDist')],
            })


In [207]:
query = """
MATCH (player:Player)-[:PLAYS_FOR]->(team:Team)
WITH player, COUNT(team) AS teamCount
WHERE teamCount > 3
RETURN player, teamCount
"""
players_errors = list()
with driver.session(database="neo4j") as session:
    r = session.run(query)
    for record in r:
        players_errors.append(record["player"]["name"])

In [224]:
# find match info they DID play in
with driver.session(database="neo4j") as session:
    query = """
    MATCH (player:Player {name: "Callum Wilson"})
    MATCH (player)-[:PLAYS_FOR]-(team:Team {name: "Newcastle United"})
    MATCH (team)-[:HOME_TEAM|AWAY_TEAM]-(m:Match)
    MATCH (player)-[stats:HOME_TEAM_PLAYER_STATS|AWAY_TEAM_PLAYER_STATS]-(m)
    RETURN DISTINCT 
      m.id AS match_id, 
      m.date, 
      m.home_team, 
      m.away_team,
      properties(stats) AS player_stats
          """

    result = session.run(query)
    count=0
    for r in result:
        print(r)
        count +=1
    print(count)

<Record match_id='GWGW12_Newcastle United_vs_West Ham United' m.date='2024-11-25' m.home_team='Newcastle United' m.away_team='West Ham United' player_stats={'assists': 0, 'tackles': 1, 'touches': 2, 'passes_completed': 2, 'blocks': 0, 'penalties': 0, 'pass_accuracy': 100.0, 'shots_on_target': 0, 'goals': 0, 'gca': 0, 'sca': 0, 'progressive_carries': 0, 'interceptions': 0, 'progressive_passes': 0, 'npxg': 0.0, 'xag': 0.0, 'carries': 3, 'shots': 0, 'xg': 0.0, 'passes_attempted': 2}>
<Record match_id='GWGW13_Crystal Palace_vs_Newcastle United' m.date='2024-11-30' m.home_team='Crystal Palace' m.away_team='Newcastle United' player_stats={'assists': 0, 'tackles': 0, 'touches': 6, 'passes_completed': 2, 'blocks': 0, 'penalties': 0, 'pass_accuracy': 66.7, 'shots_on_target': 0, 'goals': 0, 'gca': 0, 'sca': 0, 'progressive_carries': 0, 'interceptions': 0, 'progressive_passes': 0, 'npxg': 0.0, 'xag': 0.0, 'carries': 6, 'shots': 0, 'xg': 0.0, 'passes_attempted': 3}>
<Record match_id='GWGW14_Newcas

In [228]:
# find match info they DID NOT play in 
with driver.session(database="neo4j") as session:
    query = """
    MATCH (team:Team {name: "Newcastle United"})
    MATCH (team)-[:HOME_TEAM|AWAY_TEAM]-(m:Match)
    WHERE NOT (m)-[:HOME_TEAM_PLAYER_STATS|AWAY_TEAM_PLAYER_STATS]-(:Player {name: "Callum Wilson"})
    RETURN DISTINCT 
    m.id AS match_id, 
    m.date, 
    m.home_team, 
    m.away_team,
    m.away_score,
    m.home_score
    """
    result = session.run(query, {"player_name": player_name})
    count=0
    for r in result:
        print(r)
        count +=1
    print(count)
    

<Record match_id='GWGW1_Newcastle United_vs_Southampton' m.date='2024-08-17' m.home_team='Newcastle United' m.away_team='Southampton' m.away_score=['0'] m.home_score=['1']>
<Record match_id='GWGW3_Newcastle United_vs_Tottenham Hotspur' m.date='2024-09-01' m.home_team='Newcastle United' m.away_team='Tottenham Hotspur' m.away_score=['1'] m.home_score=['2']>
<Record match_id='GWGW6_Newcastle United_vs_Manchester City' m.date='2024-09-28' m.home_team='Newcastle United' m.away_team='Manchester City' m.away_score=['1'] m.home_score=['1']>
<Record match_id='GWGW8_Newcastle United_vs_Brighton & Hove Albion' m.date='2024-10-19' m.home_team='Newcastle United' m.away_team='Brighton & Hove Albion' m.away_score=['1'] m.home_score=['0']>
<Record match_id='GWGW10_Newcastle United_vs_Arsenal' m.date='2024-11-02' m.home_team='Newcastle United' m.away_team='Arsenal' m.away_score=['0'] m.home_score=['1']>
<Record match_id='GWGW16_Newcastle United_vs_Leicester City' m.date='2024-12-14' m.home_team='Newcas